# Sweep da Frequência de Corte do FIR em MobileNetV3Large

## Motivação

Em experimento anterior comparamos `DepthwiseConv2D` original × FIR fixo × wavelet
Daubechies. O **FIR Hamming separável** apresentou os melhores resultados em
acurácia e latência. Naquela rodada o `cutoff` foi fixado em **0.4** (40% da
frequência de Nyquist) — um valor arbitrário escolhido sem otimização.

**Objetivo deste experimento:** varrer sistematicamente o `cutoff` do FIR no
intervalo válido `(0, 1)` em **19 pontos** de `0.05` a `0.95` (passo `0.05`),
para identificar a faixa onde o substituto fixo iguala ou supera o baseline,
e onde ele degrada (cutoff alto → quase passa-tudo; cutoff baixo → perde
demais sinal). A avaliação a jusante usa três classificadores (MLP, XGBoost,
SVM RBF) sobre o vetor de features de 1280-D pós-GAP, para que o resultado
não dependa de uma única cabeça.

A separabilidade exata (kernel rank-1, produto externo $\mathbf{v}\mathbf{v}^\top$)
e a eliminação de pesos treináveis continuam sendo a justificativa de
velocidade — independem do `cutoff` escolhido.

In [ ]:
# ----------------------------------------------------------------------------
# Stack: TF 2 + Keras (FE e MLP), XGBoost, scikit-learn (SVM, métricas)
# ----------------------------------------------------------------------------
import os
import time
import pickle
import joblib
import numpy as np
import pandas as pd
import scipy.signal

import tensorflow as tf
import keras.backend as K

from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import (
    Input, Dense, Dropout, GlobalAveragePooling2D,
    DepthwiseConv2D, BatchNormalization,
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score as sklearn_f1
from sklearn.decomposition import PCA

from tqdm import tqdm

## Configuração do Experimento

In [ ]:
# ----------------------------------------------------------------------------
# Dados
# ----------------------------------------------------------------------------
DISPOSITIVOS = ['fridge']
IMAGENS      = ['tbg']
FOLDER_I     = 'pickle_data'

BATCH        = 32

# ----------------------------------------------------------------------------
# Sweep da frequência de corte do FIR
# ----------------------------------------------------------------------------
# `cutoff` em scipy.signal.firwin é a frequência normalizada de Nyquist:
# valores em (0, 1). Valor mais baixo  → passa-baixas mais agressivo (mantém
# só DC / componentes de menor frequência); valor mais alto → passa quase
# todo o sinal (no limite 1.0, é praticamente identidade).
FIR_CUTOFF_SWEEP = np.round(np.arange(0.05, 1.00, 0.05), 2).tolist()
# → [0.05, 0.10, 0.15, ..., 0.95]  (19 pontos)

# Quais configurações de #blocos substituídos varrer.
# Mantemos 1 e 3 (mesmas usadas no experimento anterior) para conseguir
# comparar diretamente. Comente uma das entradas se quiser reduzir o custo.
SWEEP_N_BLOCKS = [1, 3]

# ----------------------------------------------------------------------------
# Variantes do FE: baseline + sweep cartesiano (n_blocks × cutoff)
# Cada variante é (filter_type, n_blocks, cutoff). `cutoff=None` no baseline.
# ----------------------------------------------------------------------------
VARIANTS = {'baseline': ('none', 0, None)}
for nb_ in SWEEP_N_BLOCKS:
    for c in FIR_CUTOFF_SWEEP:
        VARIANTS[f'fir_{nb_}block_c{c:.2f}'] = ('fir', nb_, float(c))

print(f'Total de variantes: {len(VARIANTS)}  '
      f'(1 baseline + {len(SWEEP_N_BLOCKS)} configs × {len(FIR_CUTOFF_SWEEP)} cutoffs)')

# ----------------------------------------------------------------------------
# Classificadores avaliados sobre as features 1280-D
# ----------------------------------------------------------------------------
MODELS = ['MLP', 'XGBoost', 'SVM']

# --- MLP ---
mlp_n_runs = 5
mlp_fit_params = {
    'batch_size': BATCH,
    'epochs':     100,
    'verbose':    0,
}
mlp_early_stopping_params = {
    'monitor':              'val_loss',
    'mode':                 'min',
    'patience':             7,
    'verbose':              0,
    'restore_best_weights': False,
}
mlp_model_checkpoint_params = {
    'monitor':         'val_accuracy',
    'mode':            'max',
    'verbose':         0,
    'save_best_only':  True,
}

# --- XGBoost ---
xgb_model_params = {
    'n_estimators':          400,
    'eval_metric':           'logloss',
    'early_stopping_rounds': 7,
}
xgb_fit_params = {
    'verbose': 0,
}

# --- SVM ---
svm_model_params = {
    'kernel': 'rbf',
    # 'probability': True,   # ative se precisar de predict_proba
}

# ----------------------------------------------------------------------------
# Parâmetros fixos do filtro FIR (numtaps e janela; o cutoff varia no sweep)
# ----------------------------------------------------------------------------
FIR_NUMTAPS = 11      # usado só quando MATCH_ORIGINAL_KSIZE=False
FIR_WINDOW  = 'hamming'

# Recomendado True — garante que o kernel substituto não fique maior que o original
MATCH_ORIGINAL_KSIZE = True

## Fundamentação Teórica: Por que separável é mais barato?

### A. Custo de uma convolução depthwise 2D

Seja $x \in \mathbb{R}^{H \times W \times C}$ a entrada e
$\mathbf{K} \in \mathbb{R}^{k \times k}$ um kernel 2D aplicado canal-a-canal
(depthwise, $\text{channel\_multiplier}=1$). O custo em multiplicações-acumulações
(MACs) é

$$
\text{MACs}_{2D} \;=\; H \cdot W \cdot C \cdot k^{2}.
$$

A `DepthwiseConv2D` do Keras/cuDNN implementa exatamente isto. Como o kernel é
**aprendido**, em geral é *full-rank* — não existe garantia de fatoração — e o
compilador precisa executar a convolução $k \times k$ densa.

### B. Quando o kernel é rank-1, a separabilidade é exata

Tanto o nosso kernel FIR 2D quanto o kernel LL da DWT 2D são, por **construção**,
o produto externo de um filtro 1D por si mesmo:

$$
\mathbf{K} \;=\; \mathbf{v}\,\mathbf{v}^{\top}, \qquad \mathbf{v} \in \mathbb{R}^{k}.
$$

Isso significa que $\mathbf{K}$ tem **rank 1** e o operador de convolução com $\mathbf{K}$
pode ser fatorado:

$$
y \;=\; x * \mathbf{K} \;=\; \bigl(x *_{h} \mathbf{v}\bigr) *_{v} \mathbf{v}
$$

onde $*_h$ é convolução só no eixo horizontal (kernel $1 \times k$) e $*_v$ é
convolução só no eixo vertical (kernel $k \times 1$). O custo passa a ser

$$
\text{MACs}_{sep} \;=\; H \cdot W \cdot C \cdot (k + k) \;=\; 2\,H W C k.
$$

### C. Razão de speedup teórica

$$
\boxed{\;\frac{\text{MACs}_{2D}}{\text{MACs}_{sep}} \;=\; \frac{k^{2}}{2k} \;=\; \frac{k}{2}\;}
$$

| $k$ | $k^{2}$ (denso) | $2k$ (separável) | speedup teórico |
|----|------------------|-------------------|-----------------|
| 2  | 4                | 4                 | 1.0×            |
| 3  | 9                | 6                 | 1.5×            |
| 5  | 25               | 10                | 2.5×            |
| 7  | 49               | 14                | 3.5×            |
| 11 | 121              | 22                | 5.5×            |

Note que para $k = 2$ não há ganho — por isso o wavelet Haar (2 taps) é o pior
candidato em termos de aceleração. Para $k \geq 3$ o speedup cresce linearmente
com $k$.

### D. Por que a `DepthwiseConv2D` não pode fazer isso

A rank-1 só vale quando o kernel é separável. Um kernel aprendido $3 \times 3$
tem 9 graus de liberdade; um kernel separável tem só $2k - 1 = 5$ (descontando
a ambiguidade de escala). A perda de capacidade é o **preço a pagar** pela
fatoração — perdemos $9 - 5 = 4$ graus de liberdade por canal. No nosso caso
isso não importa porque o kernel é **fixo**, projetado a priori como passa-baixas.

### E. Por que o MobileNetV3Large é um alvo natural

Suas 15 camadas `DepthwiseConv2D` têm kernel $3 \times 3$ (3 primeiras) ou
$5 \times 5$ (demais). Substituindo as primeiras camadas — que atuam em alta
resolução ($112 \times 112$ e $56 \times 56$) e portanto têm o maior $H \cdot W$ —
maximizamos o impacto absoluto da redução de complexidade.

## Camada Customizada `FIRDepthwise`

A camada implementa a substituição:

- **`FIRDepthwise`** — Filtro FIR 2D passa-baixas projetado via janela de Hamming.
  O kernel 1D é $\mathbf{v} = \texttt{firwin}(\text{numtaps}, \text{cutoff}, \text{window})$
  do `scipy.signal`. O `cutoff` é o **parâmetro varrido** neste experimento.

Arquitetura interna:

1. No `build`, materializa dois pesos não-treináveis: um kernel horizontal
   $\mathbf{v}_h$ de shape $(1, k, C, 1)$ e um vertical $\mathbf{v}_v$ de shape
   $(k, 1, C, 1)$, ambos derivados do mesmo $\mathbf{v}$ 1D.
2. No `call`, aplica duas `tf.nn.depthwise_conv2d` sequenciais — uma na
   horizontal (com stride na largura) e outra na vertical (com stride na altura).
   O stride é aplicado **dentro** da convolução, sem `AveragePooling2D` posterior.

In [ ]:
class FIRDepthwise(tf.keras.layers.Layer):
    """
    Camada que substitui `DepthwiseConv2D` por um filtro FIR 2D passa-baixas
    fixo (não-treinável), aplicado como duas convoluções 1D separáveis.

    Definição matemática
    --------------------
    Seja v ∈ R^k o filtro FIR 1D obtido por

        v = firwin(numtaps=k, cutoff, window)        (scipy.signal)

    Construímos o kernel 2D K = v · vᵀ (rank 1, separável por construção) e
    aplicamos a convolução depthwise via fatoração:

        y = (x  *_h  v)  *_v  v        (linha → coluna)

    onde *_h e *_v denotam convolução 1D no eixo horizontal e vertical,
    respectivamente. Esta é a definição clássica de convolução separável e
    é matematicamente idêntica a uma convolução 2D com kernel K.

    Complexidade
    ------------
    Para entrada (H, W, C) e kernel k:
        - 2D denso (equivalente):  H·W·C·k²  MACs
        - Separável (esta classe): H·W·C·2k  MACs
        - Speedup teórico:         k/2

    Parâmetros
    ----------
    numtaps : int          tamanho do filtro 1D (deve ser ímpar para fase linear)
    cutoff  : float        frequência normalizada de Nyquist em (0, 1)
                           — PARÂMETRO VARRIDO NESTE EXPERIMENTO
    window  : str          janela passada a scipy.signal.firwin ('hamming' default)
    strides : (int, int)   stride aplicado dentro da convolução

    Entrada : (B, H, W, C)
    Saída   : (B, ⌈H/sh⌉, ⌈W/sw⌉, C)     (padding='SAME')
    """

    def __init__(self, numtaps=11, cutoff=0.4,
                 window='hamming', strides=(1, 1), **kwargs):
        super().__init__(**kwargs)
        self.numtaps = int(numtaps)
        self.cutoff  = float(cutoff)
        self.window  = window
        self.strides = (int(strides[0]), int(strides[1]))

    def build(self, input_shape):
        C = int(input_shape[-1])

        # Projeta filtro FIR 1D — linear-phase, janela Hamming, passa-baixas
        fir_1d = scipy.signal.firwin(
            numtaps=self.numtaps,
            cutoff=self.cutoff,
            window=self.window,
        ).astype(np.float32)

        # Materializa kernels 1D no layout esperado por depthwise_conv2d
        #   horizontal: (1, k, C, 1)  — uma linha de k coeficientes
        #   vertical:   (k, 1, C, 1)  — uma coluna de k coeficientes
        # Mesmos coeficientes são tiled em todos os C canais.
        kh = np.tile(fir_1d.reshape(1, -1, 1, 1), (1, 1, C, 1))
        kv = np.tile(fir_1d.reshape(-1, 1, 1, 1), (1, 1, C, 1))

        # Registra como pesos não-treináveis (serialização via ModelCheckpoint)
        self.k_h = self.add_weight(
            name='fir_kernel_h', shape=kh.shape,
            initializer=tf.keras.initializers.Constant(kh), trainable=False,
        )
        self.k_v = self.add_weight(
            name='fir_kernel_v', shape=kv.shape,
            initializer=tf.keras.initializers.Constant(kv), trainable=False,
        )
        super().build(input_shape)

    def call(self, x, training=None):
        sh, sw = self.strides

        # Passada horizontal — stride aplicado na largura
        x = tf.nn.depthwise_conv2d(
            x, self.k_h,
            strides=[1, 1, sw, 1],     # NHWC: (batch, height, width, channel)
            padding='SAME',
        )
        # Passada vertical — stride aplicado na altura
        x = tf.nn.depthwise_conv2d(
            x, self.k_v,
            strides=[1, sh, 1, 1],
            padding='SAME',
        )
        return x

    def get_config(self):
        return {
            **super().get_config(),
            'numtaps': self.numtaps,
            'cutoff':  self.cutoff,
            'window':  self.window,
            'strides': self.strides,
        }


CUSTOM_OBJECTS = {
    'FIRDepthwise': FIRDepthwise,
}

## Funções Auxiliares

In [ ]:
def load_data(disp, img, folder=FOLDER_I):
    """Carrega splits train/val/test de um dispositivo e tipo de imagem."""
    load = lambda fname: pickle.load(open(fname, 'rb'))
    X_tr = load(f'{folder}/X_{img}_train({disp}).pickle')
    y_tr = load(f'{folder}/y_train({disp}).pickle')
    X_va = load(f'{folder}/X_{img}_val({disp}).pickle')
    y_va = load(f'{folder}/y_val({disp}).pickle')
    X_te = load(f'{folder}/X_{img}_test({disp}).pickle')
    y_te = load(f'{folder}/y_test({disp}).pickle')
    return X_tr, y_tr, X_va, y_va, X_te, y_te


def model_size_mb(model):
    """Tamanho do modelo Keras em MB, assumindo float32 (4 bytes/parâmetro)."""
    n_params = sum(tf.size(w).numpy() for w in model.weights)
    return n_params * 4 / (1024 ** 2)


def count_params(model):
    """Retorna (n_treináveis, n_não-treináveis) de um modelo Keras."""
    trainable     = sum(tf.size(w).numpy() for w in model.trainable_weights)
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    return trainable, non_trainable


def measure_latency_ms(model, x_sample, n_warmup=5, n_reps=20):
    """
    Latência mediana de inferência em milissegundos (batch completo).

    Implementação:
      - Envolve a chamada em `tf.function(jit_compile=True)` para fusão XLA.
      - Warmup descartado para excluir o custo de compilação JIT.
      - Mediana em vez de média para robustez a outliers.
    """
    @tf.function(jit_compile=True)
    def _infer(x):
        return model(x, training=False)

    for _ in range(n_warmup):
        _ = _infer(x_sample)

    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = _infer(x_sample)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))


def build_mlp_head(feat_dim):
    """
    Cabeça MLP padrão deste projeto: Dense(64) → Dropout → Dense(64) → Dropout
    → Dense(1, sigmoid). Treina sobre features 1280-D pré-extraídas pelo FE.
    """
    mlp_input  = Input(shape=(feat_dim,))
    x          = Dense(64, activation='relu')(mlp_input)
    x          = Dropout(0.25)(x)
    x          = Dense(64, activation='relu')(x)
    x          = Dropout(0.25)(x)
    mlp_output = Dense(1, activation='sigmoid')(x)

    mlp = Model(inputs=mlp_input, outputs=mlp_output)
    mlp.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return mlp

## Construção do Extrator de Features (FE)

In [ ]:
def build_feature_extractor(input_shape, filter_type='none', n_blocks=0,
                            cutoff=0.4, name=None):
    """
    Constrói o FE baseado em MobileNetV3Large com substituição opcional de
    DepthwiseConv2D por FIR nos n_blocks primeiros blocos.

    Parâmetros
    ----------
    input_shape : tuple
        Shape de entrada (H, W, C).
    filter_type : {'none', 'fir'}
        - 'none' → MobileNetV3Large original (baseline).
        - 'fir'  → substitui primeiros n_blocks por FIRDepthwise.
    n_blocks : int
        Quantidade de DepthwiseConv2D iniciais a substituir.
    cutoff : float
        Frequência normalizada de Nyquist (0, 1) passada à FIRDepthwise.
        Ignorado quando filter_type='none'.

    Saída : Model com output_shape = (None, 1280) — vetor de features
    pós-GlobalAveragePooling2D, com todo o backbone congelado.
    """
    base = MobileNetV3Large(
        input_shape=input_shape, weights='imagenet', include_top=False,
    )

    # Baseline sem substituição
    if n_blocks == 0 or filter_type == 'none':
        for layer in base.layers:
            layer.trainable = False
        out = GlobalAveragePooling2D()(base.output)
        return Model(inputs=base.input, outputs=out,
                     name=name or 'FE_baseline')

    # Caminho com substituição
    counter = [0]

    def clone_fn(layer):
        """Troca as primeiras n_blocks DepthwiseConv2D por FIRDepthwise."""
        if isinstance(layer, DepthwiseConv2D):
            counter[0] += 1
            if counter[0] <= n_blocks:
                cfg     = layer.get_config()
                strides = cfg.get('strides', (1, 1))
                ksize   = cfg.get('kernel_size', (3, 3))
                k       = int(ksize[0])
                layer_name = f'{filter_type}_{counter[0]}'

                numtaps = k if MATCH_ORIGINAL_KSIZE else FIR_NUMTAPS
                return FIRDepthwise(
                    numtaps=numtaps,
                    cutoff=cutoff,
                    window=FIR_WINDOW,
                    strides=strides,
                    name=layer_name,
                )
        return layer

    cloned = tf.keras.models.clone_model(base, clone_function=clone_fn)

    # Copia pesos ImageNet das camadas não-substituídas
    for base_layer in base.layers:
        try:
            cloned_layer = cloned.get_layer(base_layer.name)
            weights = base_layer.get_weights()
            if weights:
                cloned_layer.set_weights(weights)
        except (ValueError, Exception):
            pass

    for layer in cloned.layers:
        layer.trainable = False

    out = GlobalAveragePooling2D()(cloned.output)
    return Model(
        inputs=cloned.input, outputs=out,
        name=name or f'FE_{filter_type}_{n_blocks}blocks_c{cutoff:.2f}',
    )

## Verificação Rápida (Smoke Test)

Confirma shapes corretos nas camadas customizadas e no FE de cada variante.

In [ ]:
_dummy_shape = (224, 224, 3)
_x = tf.random.uniform((2, *_dummy_shape))

# --- FIRDepthwise: stride=1, kernel padrão (11 taps), cutoff padrão ---
_fir = FIRDepthwise(strides=(1, 1))
_fir.build((None, *_dummy_shape))
assert _fir(_x).shape == _x.shape
print('FIRDepthwise OK:', _fir(_x).shape)

# --- FIRDepthwise: stride=2 ---
_fir_s2 = FIRDepthwise(strides=(2, 2))
_fir_s2.build((None, *_dummy_shape))
assert _fir_s2(_x).shape == (2, 112, 112, 3)
print('FIRDepthwise stride=2 OK:', _fir_s2(_x).shape)

# --- FIRDepthwise: numtaps=3 (matching 3x3 original) ---
_fir_3 = FIRDepthwise(numtaps=3, strides=(1, 1))
_fir_3.build((None, *_dummy_shape))
assert _fir_3(_x).shape == _x.shape
print('FIRDepthwise numtaps=3 OK:', _fir_3(_x).shape)

# --- FIRDepthwise: cutoffs extremos do sweep (0.05 e 0.95) ---
for _c in (0.05, 0.95):
    _f = FIRDepthwise(numtaps=3, cutoff=_c, strides=(1, 1))
    _f.build((None, *_dummy_shape))
    assert _f(_x).shape == _x.shape
    print(f'FIRDepthwise cutoff={_c} OK: {_f(_x).shape}')

# --- Output shape do FE em algumas variantes representativas ---
print('\nOutput shapes dos FEs (amostragem do sweep):')
sample_keys = ['baseline'] + [k for k in VARIANTS if k != 'baseline'][:3] + \
              [k for k in VARIANTS if k != 'baseline'][-3:]
for vname in sample_keys:
    ftype, nblocks, c = VARIANTS[vname]
    fe = build_feature_extractor(_dummy_shape, ftype, nblocks,
                                 cutoff=(c if c is not None else 0.4))
    assert fe.output_shape == (None, 1280)
    print(f'  {vname}: {fe.output_shape}  ✓')
    K.clear_session()

print('\nTodos os smoke tests passaram.')

## Experimento Principal — Sweep de Cutoff

**Estrutura:** para cada variante (baseline + cada par `(n_blocks, cutoff)` do
sweep), extraímos as features de treino/val/teste uma única vez (o FE é
totalmente congelado e determinístico), e em seguida treinamos três
classificadores sobre essas features:

- **MLP** — `mlp_n_runs = 5` execuções com inicialização aleatória; reporta média ± desvio
- **XGBoost** — 1 execução, early stopping no conjunto de validação
- **SVM (RBF)** — 1 execução, treina em (train ∪ val) por design

A latência reportada é a do FE (idêntica para os três classificadores de uma
mesma variante, já que o classificador opera sobre vetores 1280-D pequenos).
A latência **não depende do cutoff** — só de `n_blocks` —, mas é medida em
cada variante para consistência.

⚠️ **Custo:** o sweep gera `1 + len(SWEEP_N_BLOCKS) × 19` variantes. Com
`SWEEP_N_BLOCKS=[1, 3]` são 39 variantes × 3 classificadores. Se quiser
encurtar o experimento, reduza `FIR_CUTOFF_SWEEP` ou deixe apenas um
`n_blocks` na célula de configuração.

In [ ]:
os.makedirs('output', exist_ok=True)
results = []

for variant_name, (filter_type, n_blocks, cutoff) in VARIANTS.items():
    print(f"\n{'='*70}")
    print(f'VARIANTE: {variant_name}')
    print(f'  filter={filter_type}, n_blocks={n_blocks}, cutoff={cutoff}')
    print(f"{'='*70}")

    for disp in DISPOSITIVOS:
        for img in IMAGENS:

            X_tr, y_tr, X_va, y_va, X_te, y_te = load_data(disp, img)
            input_shape = X_tr.shape[1:]

            # ---------- 1. Constrói FE e extrai features uma única vez ------
            fe = build_feature_extractor(
                input_shape, filter_type, n_blocks,
                cutoff=(cutoff if cutoff is not None else 0.4),
            )
            fe_size_mb       = model_size_mb(fe)
            n_train_fe, n_frozen_fe = count_params(fe)

            print(f'  [{disp}-{img}] Extraindo features (FE)...')
            feats_tr = fe.predict(X_tr, batch_size=BATCH, verbose=0)
            feats_va = fe.predict(X_va, batch_size=BATCH, verbose=0)
            feats_te = fe.predict(X_te, batch_size=BATCH, verbose=0)

            # Latência do FE (a mesma para os 3 classificadores)
            fe_latency_ms = measure_latency_ms(fe, X_te[:BATCH])

            feat_dim = feats_tr.shape[1]
            del fe
            K.clear_session()

            print(f'  Features: shape={feats_tr.shape}, FE_latency={fe_latency_ms:.1f}ms')

            # ---------- 2. Treina cada classificador sobre as features ------
            for model_choice in MODELS:
                run_accs, run_f1s = [], []

                # ---------- MLP: mlp_n_runs execuções ----------
                if model_choice == 'MLP':
                    for run in tqdm(range(mlp_n_runs),
                                    desc=f'  {model_choice}',
                                    leave=False):
                        ckpt = (
                            f'ckpt_fir_sweep/{variant_name}/{model_choice}/'
                            f'{disp}/{img}/run{run}/model.keras'
                        )
                        os.makedirs(os.path.dirname(ckpt), exist_ok=True)

                        mlp = build_mlp_head(feat_dim)
                        mlp.fit(
                            feats_tr, y_tr,
                            validation_data=(feats_va, y_va),
                            **mlp_fit_params,
                            callbacks=[
                                EarlyStopping(**mlp_early_stopping_params),
                                ModelCheckpoint(ckpt, **mlp_model_checkpoint_params),
                            ],
                        )
                        best = tf.keras.models.load_model(ckpt)
                        preds = (
                            best.predict(feats_te, batch_size=BATCH, verbose=0) > 0.5
                        ).astype(int).flatten()
                        run_accs.append(float(np.mean(preds == y_te)))
                        run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))

                        del mlp, best
                        K.clear_session()

                # ---------- XGBoost: 1 execução com early stopping no val ----
                elif model_choice == 'XGBoost':
                    print(f'  Treinando {model_choice}...')
                    xgb = XGBClassifier(**xgb_model_params)
                    xgb.fit(
                        feats_tr, y_tr,
                        eval_set=[(feats_va, y_va)],
                        **xgb_fit_params,
                    )
                    ckpt = (
                        f'ckpt_fir_sweep/{variant_name}/{model_choice}/'
                        f'{disp}/{img}/model.json'
                    )
                    os.makedirs(os.path.dirname(ckpt), exist_ok=True)
                    xgb.save_model(ckpt)

                    preds = xgb.predict(feats_te).astype(int)
                    run_accs.append(float(accuracy_score(y_te, preds)))
                    run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))

                # ---------- SVM RBF: 1 execução, treina em (train ∪ val) -----
                elif model_choice == 'SVM':
                    print(f'  Treinando {model_choice}...')
                    feats_tr_full = np.vstack((feats_tr, feats_va))
                    y_tr_full     = np.concatenate((y_tr, y_va))

                    svm = SVC(**svm_model_params)
                    svm.fit(feats_tr_full, y_tr_full)

                    ckpt = (
                        f'ckpt_fir_sweep/{variant_name}/{model_choice}/'
                        f'{disp}/{img}/model.joblib'
                    )
                    os.makedirs(os.path.dirname(ckpt), exist_ok=True)
                    joblib.dump(svm, ckpt)

                    preds = svm.predict(feats_te).astype(int)
                    run_accs.append(float(accuracy_score(y_te, preds)))
                    run_f1s.append(float(sklearn_f1(y_te, preds, average='macro')))

                # ---------- Salva resultados do (variante, modelo) -----------
                row = {
                    'variant':           variant_name,
                    'filter_type':       filter_type,
                    'n_blocks':          n_blocks,
                    'cutoff':            cutoff,             # NOVO: coluna do sweep
                    'model_choice':      model_choice,
                    'device':            disp,
                    'image':             img,
                    'acc_mean':          float(np.mean(run_accs)),
                    'acc_std':           float(np.std(run_accs)),
                    'f1_mean':           float(np.mean(run_f1s)),
                    'f1_std':            float(np.std(run_f1s)),
                    'fe_latency_ms':     fe_latency_ms,
                    'fe_size_mb':        fe_size_mb,
                    'n_train_fe':        n_train_fe,
                    'n_frozen_fe':       n_frozen_fe,
                    'n_runs':            len(run_accs),
                }
                results.append(row)

                acc_str = (
                    f'{np.mean(run_accs):.3f}±{np.std(run_accs):.3f}'
                    if len(run_accs) > 1 else f'{run_accs[0]:.3f}'
                )
                print(f'    {model_choice:8s} | acc={acc_str} | f1={np.mean(run_f1s):.3f}')

                # Salva CSV incrementalmente — útil em sweeps longos para
                # poder analisar resultados parciais sem perder progresso.
                pd.DataFrame(results).to_csv(
                    'output/fir_cutoff_sweep_results.csv', index=False,
                )

            del feats_tr, feats_va, feats_te

df_results = pd.DataFrame(results)
df_results.to_csv('output/fir_cutoff_sweep_results.csv', index=False)
print('\nResultados salvos -> output/fir_cutoff_sweep_results.csv')
df_results

## Análise Quantitativa

In [ ]:
# Carrega resultados (caso o notebook seja reiniciado)
df_results = pd.read_csv('output/fir_cutoff_sweep_results.csv')

# Resumo agregado por (variante, classificador): média sobre dispositivos × imagens
summary = df_results.groupby(
    ['variant', 'filter_type', 'n_blocks', 'cutoff', 'model_choice'],
    dropna=False,
).agg(
    acc_mean=('acc_mean', 'mean'),
    acc_std=('acc_std', 'mean'),
    f1_mean=('f1_mean', 'mean'),
    fe_latency_ms=('fe_latency_ms', 'mean'),
    fe_size_mb=('fe_size_mb', 'first'),
).reset_index()

# ----------------------------------------------------------------------------
# Tabela 1 — Melhor cutoff por (n_blocks, classificador) em acurácia
# ----------------------------------------------------------------------------
fir_only = summary[summary['filter_type'] == 'fir'].copy()
best_per_config = (
    fir_only.sort_values('acc_mean', ascending=False)
            .groupby(['n_blocks', 'model_choice'], as_index=False)
            .first()
            .loc[:, ['n_blocks', 'model_choice', 'cutoff',
                     'acc_mean', 'f1_mean', 'fe_latency_ms']]
            .sort_values(['n_blocks', 'model_choice'])
)
print('=== Melhor cutoff por (n_blocks × classificador) ===')
print(best_per_config.round(4).to_string(index=False))

# ----------------------------------------------------------------------------
# Tabela 2 — Baseline para referência
# ----------------------------------------------------------------------------
baseline = summary[summary['variant'] == 'baseline'][
    ['model_choice', 'acc_mean', 'f1_mean', 'fe_latency_ms']
]
print('\n=== Baseline (DepthwiseConv2D original) ===')
print(baseline.round(4).to_string(index=False))

# ----------------------------------------------------------------------------
# Tabela 3 — Curva completa (pivot: cutoff × classificador) por n_blocks
# ----------------------------------------------------------------------------
for nb_ in sorted(fir_only['n_blocks'].unique()):
    sub = fir_only[fir_only['n_blocks'] == nb_]
    pivot = sub.pivot(index='cutoff', columns='model_choice', values='acc_mean')
    pivot = pivot[[c for c in ['MLP', 'XGBoost', 'SVM'] if c in pivot.columns]]
    print(f'\n=== Acurácia × cutoff (n_blocks={int(nb_)}) ===')
    print(pivot.round(4).to_string())

In [ ]:
from matplotlib import pyplot as plt

colors = {'MLP': 'steelblue', 'XGBoost': 'darkorange', 'SVM': 'seagreen'}

# Linhas baseline (uma horizontal por classificador, para servir de referência)
baseline_by_model = (
    summary[summary['variant'] == 'baseline']
    .set_index('model_choice')[['acc_mean', 'f1_mean']]
)

# ----------------------------------------------------------------------------
# Plot 1 — Curva acurácia × cutoff, um subplot por n_blocks
# ----------------------------------------------------------------------------
n_blocks_vals = sorted(fir_only['n_blocks'].unique())
fig, axes = plt.subplots(
    1, len(n_blocks_vals),
    figsize=(7 * len(n_blocks_vals), 5),
    sharey=True, squeeze=False,
)

for ax, nb_ in zip(axes[0], n_blocks_vals):
    sub = fir_only[fir_only['n_blocks'] == nb_]
    for mc in ['MLP', 'XGBoost', 'SVM']:
        s = sub[sub['model_choice'] == mc].sort_values('cutoff')
        if s.empty:
            continue
        ax.plot(
            s['cutoff'], s['acc_mean'],
            marker='o', linewidth=1.5,
            label=f'{mc} (FIR)', color=colors.get(mc),
        )
        # Linha tracejada com o baseline correspondente
        if mc in baseline_by_model.index:
            base_acc = baseline_by_model.loc[mc, 'acc_mean']
            ax.axhline(
                base_acc, linestyle='--', linewidth=1, alpha=0.6,
                color=colors.get(mc),
                label=f'{mc} (baseline)',
            )
    ax.set_xlabel('cutoff (normalizado, Nyquist=1)')
    ax.set_title(f'FIR substituindo {int(nb_)} bloco(s) — acurácia × cutoff')
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', fontsize=8, ncol=2)

axes[0, 0].set_ylabel('Acurácia média (teste)')
plt.tight_layout()
plt.savefig('output/fir_cutoff_sweep_acc.png', dpi=150, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------------------
# Plot 2 — F1-macro × cutoff (mesma estrutura)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(
    1, len(n_blocks_vals),
    figsize=(7 * len(n_blocks_vals), 5),
    sharey=True, squeeze=False,
)
for ax, nb_ in zip(axes[0], n_blocks_vals):
    sub = fir_only[fir_only['n_blocks'] == nb_]
    for mc in ['MLP', 'XGBoost', 'SVM']:
        s = sub[sub['model_choice'] == mc].sort_values('cutoff')
        if s.empty:
            continue
        ax.plot(
            s['cutoff'], s['f1_mean'],
            marker='o', linewidth=1.5,
            label=f'{mc} (FIR)', color=colors.get(mc),
        )
        if mc in baseline_by_model.index:
            base_f1 = baseline_by_model.loc[mc, 'f1_mean']
            ax.axhline(
                base_f1, linestyle='--', linewidth=1, alpha=0.6,
                color=colors.get(mc),
                label=f'{mc} (baseline)',
            )
    ax.set_xlabel('cutoff (normalizado, Nyquist=1)')
    ax.set_title(f'FIR substituindo {int(nb_)} bloco(s) — F1 × cutoff')
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', fontsize=8, ncol=2)

axes[0, 0].set_ylabel('F1-macro')
plt.tight_layout()
plt.savefig('output/fir_cutoff_sweep_f1.png', dpi=150, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------------------
# Plot 3 — Latência do FE por variante (depende só de n_blocks, não do cutoff)
# Mostra baseline + média ± std da latência observada em cada n_blocks
# ----------------------------------------------------------------------------
lat_agg = (
    summary.groupby(['filter_type', 'n_blocks'])['fe_latency_ms']
           .agg(['mean', 'std', 'min', 'max'])
           .reset_index()
)
print('\n=== Latência do FE por configuração (ms) ===')
print(lat_agg.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
labels, vals, errs = [], [], []
for _, row in lat_agg.iterrows():
    label = ('baseline' if row['filter_type'] == 'none'
             else f"fir_{int(row['n_blocks'])}block")
    labels.append(label)
    vals.append(row['mean'])
    errs.append(row['std'] if not np.isnan(row['std']) else 0.0)

ax.barh(labels, vals, xerr=errs, color='darkorange', edgecolor='black')
ax.set_xlabel('Latência do FE (ms, mediana por variante; barra = std no sweep de cutoff)')
ax.set_title('Latência do FE — depende de n_blocks, não do cutoff')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('output/fir_cutoff_sweep_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## Visualização das Features Extraídas

Duas visualizações complementares — ambas operam apenas sobre o FE, ou seja,
são independentes da escolha do classificador:

- **A.** Mapas de features após a primeira camada substituída, comparando
  baseline e três cutoffs representativos do sweep (baixo, médio, alto).
- **B.** Projeção PCA 2D dos vetores globais (1280-D) para o conjunto de teste,
  também comparando baseline com três cutoffs.

In [ ]:
from matplotlib import pyplot as plt

# Carrega uma amostra de teste para visualização
_X_tr, _y_tr, _X_va, _y_va, X_te_viz, y_te_viz = load_data(DISPOSITIVOS[0], IMAGENS[0])
_input_shape = X_te_viz.shape[1:]
print(f'Conjunto de teste: {X_te_viz.shape}, classes únicas = {np.unique(y_te_viz)}')


def get_first_substituted_output(filter_type, n_blocks, cutoff, x_sample):
    """
    Retorna a saída da primeira camada que sofreu substituição em cada
    variante. Para o baseline, retorna a saída da primeira DepthwiseConv2D
    original (mesmo ponto topológico da rede).
    """
    fe = build_feature_extractor(
        _input_shape, filter_type, n_blocks,
        cutoff=(cutoff if cutoff is not None else 0.4),
    )
    if filter_type == 'none':
        target = next(l for l in fe.layers if isinstance(l, DepthwiseConv2D))
    else:
        target = fe.get_layer('fir_1')
    sub = Model(inputs=fe.input, outputs=target.output)
    return sub(x_sample, training=False).numpy()


SAMPLE_IDX = 0
N_CHANNELS = 4

# Compara baseline com 3 cutoffs representativos (baixo / médio / alto)
viz_variants = [
    ('baseline (DepthwiseConv2D 3×3)', 'none', 0, None),
    ('FIR cutoff=0.10 (passa-baixas agressivo)', 'fir', 1, 0.10),
    ('FIR cutoff=0.40 (valor original)',         'fir', 1, 0.40),
    ('FIR cutoff=0.90 (quase passa-tudo)',       'fir', 1, 0.90),
]

x_in = X_te_viz[SAMPLE_IDX:SAMPLE_IDX + 1]
feature_maps = {}
for label, ftype, nblocks, c in viz_variants:
    fmap = get_first_substituted_output(ftype, nblocks, c, x_in)
    feature_maps[label] = fmap[0]
    print(f'{label:50s} → feature map shape {fmap.shape}')
    K.clear_session()

# --- Plot ---
fig, axes = plt.subplots(
    len(viz_variants) + 1, N_CHANNELS,
    figsize=(N_CHANNELS * 2.8, (len(viz_variants) + 1) * 2.8),
)

# Linha 0: imagem de entrada
for j in range(N_CHANNELS):
    ax = axes[0, j]
    if j == N_CHANNELS // 2:
        img = x_in[0]
        img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img_norm)
        ax.set_title(f'Entrada (#{SAMPLE_IDX}, classe={int(y_te_viz[SAMPLE_IDX])})',
                     fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

# Linhas seguintes: feature maps
for i, (label, fmap) in enumerate(feature_maps.items(), start=1):
    for j in range(N_CHANNELS):
        ax = axes[i, j]
        ax.imshow(fmap[:, :, j], cmap='viridis')
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(label, rotation=0, fontsize=9, labelpad=100,
                          ha='right', va='center')
        ax.set_title(f'canal {j}', fontsize=8)

plt.suptitle('Mapas de features — efeito do cutoff do FIR no primeiro bloco',
             fontsize=12, y=1.00)
plt.tight_layout()
plt.savefig('output/feature_maps_by_cutoff.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def get_global_features(filter_type, n_blocks, cutoff, X_data):
    """Vetor 1280-D pós-GAP para cada amostra (FE é frozen e determinístico)."""
    fe = build_feature_extractor(
        _input_shape, filter_type, n_blocks,
        cutoff=(cutoff if cutoff is not None else 0.4),
    )
    feats = fe.predict(X_data, batch_size=BATCH, verbose=0)
    K.clear_session()
    return feats


pca_variants = [
    ('baseline',           'none', 0, None),
    ('fir_1block_c0.10',   'fir',  1, 0.10),
    ('fir_1block_c0.40',   'fir',  1, 0.40),
    ('fir_1block_c0.90',   'fir',  1, 0.90),
]

fig, axes = plt.subplots(1, len(pca_variants), figsize=(5 * len(pca_variants), 5))

for ax, (label, ftype, nblocks, c) in zip(axes, pca_variants):
    feats = get_global_features(ftype, nblocks, c, X_te_viz)
    proj  = PCA(n_components=2).fit_transform(feats)
    for cls in np.unique(y_te_viz):
        mask = (y_te_viz == cls)
        ax.scatter(
            proj[mask, 0], proj[mask, 1],
            label=f'classe {int(cls)}', alpha=0.6, s=25,
            edgecolors='white', linewidth=0.3,
        )
    ax.set_title(f'{label}\n(1280-D → PCA 2D)', fontsize=10)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Separabilidade no espaço de features globais (teste) — efeito do cutoff',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('output/global_features_pca_by_cutoff.png', dpi=150, bbox_inches='tight')
plt.show()

## Discussão

### O que esperar do sweep do cutoff?

O parâmetro `cutoff` controla quanto conteúdo de alta frequência o filtro
passa-baixas deixa passar. Como o `cutoff` é normalizado por Nyquist:

- **Cutoff próximo de 0** (ex.: 0.05): o FIR atenua quase tudo exceto a
  componente DC. O feature map fica praticamente uniforme — perdemos
  informação útil para discriminar dispositivos.
- **Cutoff médio** (faixa 0.2 – 0.5): preserva contornos suaves e estrutura
  de baixa frequência da assinatura — é a faixa onde a hipótese de que a
  primeira `DepthwiseConv2D` opera como passa-baixas deve render melhor
  acurácia.
- **Cutoff próximo de 1** (ex.: 0.9): o FIR passa quase todo o sinal — vira
  quase identidade. A acurácia tende a aproximar-se do *passthrough* (sem
  filtragem), o que **pode ou não** ser pior que o baseline aprendido, já
  que o baseline também explora alguma estrutura espacial.

A curva esperada é, portanto, em formato de **sino assimétrico** ou **platô
seguido de queda**: subida rápida saindo do cutoff baixo, platô no meio,
comportamento ambíguo no cutoff alto.

### Por que avaliar com 3 classificadores diferentes?

A acurácia final é função de (FE × classificador). Reportando apenas com MLP,
poderia-se argumentar que o resultado depende da expressividade da cabeça
neural. Avaliando também com **XGBoost** (boosting de árvores) e **SVM RBF**
(margem máxima em espaço RKHS), trianglamos a qualidade das features:

- Se as três cabeças produzirem acurácia semelhante para um mesmo cutoff, é
  evidência de que a informação discriminativa está nas features.
- Se o ótimo do `cutoff` for diferente para classificadores diferentes,
  isso indica que a melhor representação depende da família de decisão
  (linear-like em RKHS × árvores × MLP).

### Latência

A latência do FE depende de `n_blocks` (quantas `DepthwiseConv2D` foram
substituídas) e do tamanho dos kernels, mas **não depende do `cutoff`** —
o custo de aplicar `tf.nn.depthwise_conv2d` é o mesmo para qualquer
valor numérico dos pesos fixos. Confirmamos isto reportando média e desvio
da latência ao longo de todo o sweep para cada `n_blocks` — o desvio deve
ser pequeno comparado à média.

### Limites desta abordagem

- **Granularidade do sweep:** 19 pontos com passo 0.05 dão uma resolução
  razoável para identificar a faixa ótima; para refinamento fino, faz
  sentido um segundo sweep com passo menor (ex.: 0.01) em volta do melhor
  ponto encontrado.
- **Interação com `n_blocks`:** o cutoff ótimo pode mudar conforme aumenta
  `n_blocks`, já que cada camada subsequente vê features mais abstratas.
  Por isso a tabela é reportada por `n_blocks` separadamente.
- **Janela e numtaps fixos:** mantivemos `window='hamming'` e `numtaps=k`
  (igual ao kernel original). Outros designs (Kaiser, Blackman) ou
  comprimentos diferentes poderiam alterar a forma da curva.

## Referências

**Arquitetura backbone:**
- Howard, A. *et al.* (2019). *Searching for MobileNetV3.* ICCV.
- Sandler, M. *et al.* (2018). *MobileNetV2: Inverted Residuals and Linear Bottlenecks.* CVPR.

**Design de filtros FIR:**
- Oppenheim, A. V., & Schafer, R. W. (2010). *Discrete-Time Signal Processing*
  (3rd ed.). Cap. 7: FIR filter design by windowing.
- Harris, F. J. (1978). *On the use of windows for harmonic analysis with
  the discrete Fourier transform.* Proceedings of the IEEE, 66(1), 51-83.

**Convoluções separáveis em redes convolucionais:**
- Mamalet, F. & Garcia, C. (2012). *Simplifying ConvNets for Fast Learning.* ICANN.
- Sironi, A. *et al.* (2015). *Learning Separable Filters.* IEEE TPAMI.

**Classificadores avaliados:**
- Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD.
- Cortes, C. & Vapnik, V. (1995). *Support-Vector Networks.* Machine Learning, 20(3).

**Compilação XLA:**
- TensorFlow XLA documentation: <https://www.tensorflow.org/xla>